# Heating Pad Temperature Analysis – Sunbeam Data
**Name:** Pete Mathew  
**Course:** (fill in)  
**Date:** 2025-09-07  
**Notebook:** HeatingPad_Analysis_PeteMathew.ipynb

This notebook demonstrates reading CSV and spreadsheet (ODS) temperature data, cleaning and transforming it, then plotting and summarizing statistics for four probe locations (P1–P4) on a heating pad sampled at 1 Hz.  


In [ ]:
# Environment check
import sys
import numpy as np
import pandas as pd
import matplotlib as mpl
import matplotlib.pyplot as plt

try:
    import scipy as sp
    SCIPY_OK = True
except Exception as e:
    SCIPY_OK = False
    print("SciPy not available:", e)

print('Python:     {:d}.{:d}'.format(sys.version_info[0], sys.version_info[1]))
print('Matplotlib: ', mpl.__version__)
print('Pandas:     ', pd.__version__)
if SCIPY_OK:
    import scipy as sp
    print('SciPy:      ', sp.__version__)
else:
    print('SciPy:      (not installed)')

# Plot defaults (no explicit colors per assignment instructions)
plt.rcParams.update({
    "figure.figsize": (10, 6),
    "axes.grid": True
})


## File Inputs & Helpers

Place your data files in the same folder as this notebook:
- `HP-test-file.csv`
- `HP-test-file.ods` (sheet: `test_file`)

> If files are missing, a **mock dataset** can be generated to let you run the notebook end-to-end.


In [ ]:
from pathlib import Path
import warnings

# Filenames
csv_file = Path('HP-test-file.csv')
ods_file = Path('HP-test-file.ods')
ods_sheet = 'test_file'  # provided in the prompt

def preview(df, n=5, name="DataFrame"):
    print(f"\n{name} shape: {df.shape}")
    display(df.head(n))


## 1) Read CSV (with Pandas)

- `parse_dates=['Date']` will automatically parse `Date` strings into datetime objects.
- We keep `Time` as text initially and build a full timestamp later.
- Column names: `['Date', 'Time', 'P1', 'P2', 'P3', 'P4']`.


In [ ]:
import pandas as pd

colnames = ['Date', 'Time', 'P1', 'P2', 'P3', 'P4']
datecol = ['Date']
df_csv = None

if csv_file.exists():
    df_csv = pd.read_csv(csv_file,
                         header=None,
                         names=colnames,
                         parse_dates=datecol,
                         infer_datetime_format=True)
    preview(df_csv, name="df_csv (raw)")
else:
    warnings.warn(f"CSV file not found: {csv_file}. You can generate mock data below.")


## 2) Read spreadsheet (with Pandas)

- Uses `engine='odf'` for `.ods` spreadsheets (requires `odfpy`).
- Sheet selection via `sheet_name='test_file'` per the prompt.


In [ ]:
df_odf = None
if ods_file.exists():
    try:
        df_odf = pd.read_excel(ods_file,
                               header=None,
                               names=colnames,
                               parse_dates=datecol,  # note: ODS typically carries date types; keep explicit
                               engine='odf',
                               sheet_name=ods_sheet,
                               skiprows=0)
        preview(df_odf, name="df_odf (raw)")
    except Exception as e:
        warnings.warn(f"Could not read ODS: {e}\nInstall with: pip install odfpy")
else:
    warnings.warn(f"ODS file not found: {ods_file}. You can generate mock data below.")


### (Optional) Generate Mock Data

If you do not have the CSV/ODS files yet, run this cell to create a small synthetic dataset that mimics 1 Hz sampling for ~5 minutes and save it as `HP-test-file.csv` so you can run the rest of the notebook.


In [ ]:
import numpy as np
import pandas as pd
from datetime import datetime, timedelta

if not csv_file.exists():
    t0 = datetime.now().replace(microsecond=0)
    n = 300  # 5 minutes at 1 Hz
    dates = [t0.date() for _ in range(n)]
    times = [(t0 + timedelta(seconds=i)).strftime("%H:%M:%S") for i in range(n)]

    # Four probes with gentle warm-up curves + noise
    base = np.linspace(25, 55, n)
    P1 = base + np.random.normal(0, 0.4, n)
    P2 = base + 1.0 + np.random.normal(0, 0.7, n)
    P3 = base - 0.5 + np.random.normal(0, 0.5, n)
    P4 = base + np.sin(np.linspace(0, 8*np.pi, n)) + np.random.normal(0, 0.6, n)

    mock = pd.DataFrame({"Date": dates, "Time": times, "P1": P1, "P2": P2, "P3": P3, "P4": P4})
    mock.to_csv(csv_file, header=False, index=False)
    print(f"Generated mock CSV: {csv_file.resolve()}")

# Reload CSV if we just created it
if df_csv is None and csv_file.exists():
    df_csv = pd.read_csv(csv_file, header=None, names=['Date','Time','P1','P2','P3','P4'], parse_dates=['Date'])
    preview(df_csv, name="df_csv (mock)")


## 3) Datetime Handling – Build a Timestamp & Elapsed Seconds

- Combine `Date` and `Time` into a full timestamp.
- Create an `elapsed_s` column based on the first timestamp (assumes 1 Hz if no drift).


In [ ]:
def add_time_columns(df):
    if df is None or df.empty:
        return df
    # Ensure strings
    df = df.copy()
    df['Date'] = pd.to_datetime(df['Date']).dt.date
    # Some files store Time as 'HH:MM:SS' strings; enforce that
    df['Time'] = df['Time'].astype(str)
    # Build Timestamp
    df['Timestamp'] = pd.to_datetime(df['Date'].astype(str) + ' ' + df['Time'], errors='coerce')
    # Elapsed seconds from the first valid timestamp
    t0 = df['Timestamp'].dropna().iloc[0]
    df['elapsed_s'] = (df['Timestamp'] - t0).dt.total_seconds()
    return df

df_csv = add_time_columns(df_csv)
if df_csv is not None:
    preview(df_csv, name="df_csv (+time)")

df_odf = add_time_columns(df_odf)
if df_odf is not None:
    preview(df_odf, name="df_odf (+time)")


## 4) Basic Plots

Single-series quick plots for P1 from CSV and ODS.


In [ ]:
# 4.a CSV P1 vs Time (string)
if df_csv is not None:
    ax = df_csv.plot(x='Time', y='P1', xlabel='Time', ylabel='Temperature (°C)', title='CSV – P1 vs Time (raw text axis)')
    plt.show()

# 4.b ODS P1 vs Time (if present)
if df_odf is not None:
    ax = df_odf.plot(x='Time', y='P1', xlabel='Time', ylabel='Temperature (°C)', title='ODS – P1 vs Time (raw text axis)')
    plt.show()


## 5) Cleaner Plots – Use Elapsed Seconds

Plot P1 (and later all probes) against `elapsed_s` to get proper numeric x-axis in seconds.


In [ ]:
# CSV P1 vs elapsed seconds
if df_csv is not None:
    ax = df_csv.plot(x='elapsed_s', y='P1', xlabel='Time (s)', ylabel='Temperature (°C)', title='CSV – P1 vs Elapsed Seconds')
    plt.show()

# Both frames on one axis (P1) if both exist
if df_csv is not None and df_odf is not None:
    fig = plt.figure()
    ax = plt.subplot(111)  # single axis (no subplots per instructions)
    df_csv.plot(x='elapsed_s', y='P1', ax=ax, label='CSV P1')
    df_odf.plot(x='elapsed_s', y='P1', ax=ax, label='ODS P1')
    ax.set_xlabel('Time (s)')
    ax.set_ylabel('Temperature (°C)')
    ax.set_title('P1 – CSV vs ODS')
    plt.show()


## 6) Fiddle with DataFrame

- Example of dropping unused columns and NaNs (CSV only here).  
> *Note:* Do **not** drop `Time`/`Date` until after building `Timestamp`/`elapsed_s`.


In [ ]:
df_work = None
if df_csv is not None:
    df_work = df_csv.copy()
    # After building time columns, original Date can be removed if desired:
    df_work.drop(columns=['Date'], inplace=True, errors='ignore')
    df_work.dropna(inplace=True)
    preview(df_work, name="df_work (clean CSV)")


## 7) Plot All Sequences (P1–P4) on One Axis


In [ ]:
if df_work is not None:
    ax = df_work.plot(x='elapsed_s', y=['P1','P2','P3','P4'],
                      xlabel='Time (s)', ylabel='Temperature (°C)',
                      title='All Probes – CSV')
    plt.show()


## 8) Useful Things to Do

- Compute statistics (min/mean/max, std).
- Rolling averages for smoothing.
- Add peak annotations.


In [ ]:
import numpy as np

def probe_stats(df, label="df"):
    probes = ['P1','P2','P3','P4']
    subset = df[probes].dropna()
    desc = subset.describe().T[['min','mean','std','max']]
    print(f"\n{label} – Summary Statistics (°C)")
    display(desc.round(3))
    return desc

if df_work is not None:
    stats = probe_stats(df_work, "CSV (clean)")

    # Rolling mean (5 s window)
    roll = df_work[['P1','P2','P3','P4']].rolling(window=5, min_periods=1).mean()
    df_roll = df_work[['elapsed_s']].join(roll)
    ax = df_roll.plot(x='elapsed_s', y=['P1','P2','P3','P4'],
                      xlabel='Time (s)', ylabel='Temperature (°C)',
                      title='All Probes – 5‑Second Rolling Mean')
    plt.show()

    # Annotate peaks for P1
    p1_idx = df_work['P1'].idxmax()
    if pd.notna(p1_idx):
        peak_t = df_work.loc[p1_idx, 'elapsed_s']
        peak_y = df_work.loc[p1_idx, 'P1']
        ax = df_work.plot(x='elapsed_s', y='P1', title='P1 with Peak Annotation',
                          xlabel='Time (s)', ylabel='Temperature (°C)')
        plt.annotate(f"Peak: {peak_y:.1f}°C @ {peak_t:.0f}s",
                     xy=(peak_t, peak_y),
                     xytext=(peak_t, peak_y + 2),
                     arrowprops=dict(arrowstyle="->"))
        plt.show()


## 9) Multiple Files – Aggregate

Read several CSV files (pattern `HP*/*.csv` or `HP*.csv`), combine them, and do per-probe statistics grouped by file.


In [ ]:
import glob
from pathlib import Path

def load_csv_file(p: Path):
    df = pd.read_csv(p, header=None, names=['Date','Time','P1','P2','P3','P4'], parse_dates=['Date'])
    df = add_time_columns(df)
    df['source_file'] = p.name
    return df

# Try a couple of common patterns (adjust as needed)
patterns = ['HP*.csv', 'data/HP*.csv', 'HP*/*.csv']
all_files = []
for pat in patterns:
    all_files.extend([Path(p) for p in glob.glob(pat)])

dfs = []
for p in all_files:
    try:
        dfs.append(load_csv_file(p))
    except Exception as e:
        print(f"Skipping {p}: {e}")

if dfs:
    df_all = pd.concat(dfs, ignore_index=True)
    print(f"Loaded {len(dfs)} files, combined shape: {df_all.shape}")
    # Group stats per file
    grp = df_all.groupby('source_file')[['P1','P2','P3','P4']].agg(['min','mean','std','max']).round(3)
    display(grp)
else:
    print("No additional CSV files found with patterns:", patterns)


## 10) Save Figures

Example of saving a figure to PNG for your report.


In [ ]:
if df_work is not None:
    ax = df_work.plot(x='elapsed_s', y=['P1','P2','P3','P4'],
                      xlabel='Time (s)', ylabel='Temperature (°C)',
                      title='All Probes – CSV (Saved)')
    out_png = Path('heating_pad_all_probes_pete.png')
    plt.savefig(out_png, dpi=300, bbox_inches='tight')
    plt.show()
    print("Saved figure to:", out_png.resolve())


## 11) What Else – Something Cool ✨

- **Derivative (°C/s)**: approximate heating rate using finite differences.  
- **Auto-segment warm‑up vs steady‑state**: detect when the derivative falls below a threshold.  


In [ ]:
import numpy as np

if df_work is not None:
    # Finite difference derivative for P1
    df_rate = df_work[['elapsed_s','P1']].dropna().copy()
    dt = np.diff(df_rate['elapsed_s'], prepend=df_rate['elapsed_s'].iloc[0])
    dt[dt == 0] = 1.0  # safeguard
    dT = np.diff(df_rate['P1'], prepend=df_rate['P1'].iloc[0])
    rate = dT / dt
    df_rate['dT_dt'] = rate

    # Simple threshold to find steady-state (rate magnitude below eps for N consecutive points)
    eps = 0.02  # °C/s
    window = 10
    below = (df_rate['dT_dt'].abs() < eps).rolling(window=window, min_periods=1).mean() == 1.0
    # First time index where condition holds
    ss_idx = below.idxmax()  # first True in a run
    ss_t = df_rate.loc[ss_idx, 'elapsed_s'] if ss_idx in df_rate.index else None

    # Plot P1 + mark steady-state
    ax = df_work.plot(x='elapsed_s', y='P1', xlabel='Time (s)', ylabel='Temperature (°C)',
                      title='P1 with Estimated Steady‑State Marker')
    if ss_t is not None and np.isfinite(ss_t):
        plt.axvline(ss_t, linestyle='--')
        plt.annotate(f"Steady‑state ~{ss_t:.0f}s", xy=(ss_t, df_work['P1'].max()),
                     xytext=(ss_t, df_work['P1'].max()+1),
                     arrowprops=dict(arrowstyle='->'))
    plt.show()

    # Plot derivative
    ax = df_rate.plot(x='elapsed_s', y='dT_dt', xlabel='Time (s)', ylabel='dT/dt (°C/s)',
                      title='Heating Rate – P1')
    plt.show()


---

### Submission Checklist
- Rename the file to include your name (already set): **HeatingPad_Analysis_PeteMathew.ipynb**.  
- Ensure your name is present in the first Markdown cell.  
- Run **Kernel → Restart & Run All** to ensure a clean, reproducible execution.  
- Upload the notebook and any figures (`.png`) to Canvas.  
